# Interactive Visualization of Kratos Meshes with PyVista

This Jupyter Notebook demonstrates how to use the `KratosMultiphysics.pyvista_utilities` module to convert Kratos `ModelPart` objects into PyVista grids for rich 3D visualization and post-processing.

**Key utilities covered:**
- `ModelPartToPyVista` — low-level conversion to `pyvista.UnstructuredGrid`
- `PlotModelPart` — high-level rendering with component selection, warp, camera presets, and composable plotters
- `ScreenshotModelPart` — off-screen screenshot export
- `CreateExtractedSurface` — outer surface extraction for 3-D meshes
- `ComputeWarpedMesh`, `CreateOrthogonalSlices`, `CreateIsosurfaces`, `CreateStreamlines`, `CreateThresholdedMesh`, `CreateVectorGlyphs`, `CreateClippedMesh`

### Step 1: Import Modules & Choose Visualization Mode

We import NumPy, PyVista, Kratos Multiphysics, and our PyVista utilities module.

> **Note:** `pyvista` is an **optional** dependency of Kratos. `pyvista_utilities` defers every
> `import pyvista` to the inside of each function, so the rest of Kratos loads even when PyVista is absent.

#### Enabling Interactive 3D Rendering
By default, this notebook renders high-quality **static** images. To enable interactive rotate/pan/zoom:
1. Install the required packages:
   ```bash
   pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
   ```
2. Un-comment and run the interactive backend lines in the cell below.


In [ ]:
import sys
import os

# Resolve the Kratos repo root relative to this notebook's location:
# notebooks/ -> python_scripts/ -> kratos/ -> <repo root>
# NOTE: Only needed for self-compiled Kratos; skip for system-wide installations.
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
kratos_root = os.path.abspath(os.path.join(notebook_dir, "..", "..", ".."))
kratos_build_path = os.path.join(kratos_root, "bin", "Release")

if kratos_build_path not in sys.path:
    sys.path.insert(0, kratos_build_path)

print(f"Kratos build path: {kratos_build_path}")

Kratos build path: /home/vicente/src/Kratos/bin/Release


In [ ]:
import numpy as np
import pyvista as pv
import KratosMultiphysics as KM
import KratosMultiphysics.pyvista_utilities as pv_utils

# Set PyVista theme colour
pv.global_theme.color = "white"

# ==========================================================================
# INTERACTIVE 3D VISUALIZATION IN JUPYTER NOTEBOOKS:
# 1. pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
# 2. Un-comment the lines below:
#    import nest_asyncio
#    nest_asyncio.apply()
#    pv.set_jupyter_backend("client")
# ==========================================================================

print("Modules imported successfully!")

Modules imported successfully!


### Step 2: Create a Kratos ModelPart with Results Data

We set up a simple 2D triangular mesh with `PRESSURE` (scalar) and `DISPLACEMENT` (vector) as historical solution-step variables.

In [ ]:
model = KM.Model()
mp = model.CreateModelPart("SimulationResults")

mp.AddNodalSolutionStepVariable(KM.DISPLACEMENT)
mp.AddNodalSolutionStepVariable(KM.PRESSURE)

mp.CreateNewNode(1, 0.0, 0.0, 0.0)
mp.CreateNewNode(2, 2.0, 0.0, 0.0)
mp.CreateNewNode(3, 0.0, 2.0, 0.0)
mp.CreateNewNode(4, 2.0, 2.0, 0.0)

for node in mp.Nodes:
    node.SetSolutionStepValue(KM.PRESSURE,     0, 15.0 * node.X + 25.0 * node.Y)
    node.SetSolutionStepValue(KM.DISPLACEMENT, 0, [0.15 * node.X, 0.15 * node.Y, 0.0])

properties = mp.GetProperties()[0]
mp.CreateNewElement("Element2D3N", 1, [1, 2, 3], properties)
mp.CreateNewElement("Element2D3N", 2, [2, 4, 3], properties)

print(f"Mesh created with {len(mp.Nodes)} nodes and {len(mp.Elements)} elements.")

Mesh created with 4 nodes and 2 elements.


### Step 3: Convert Kratos ModelPart to a PyVista Grid

`ModelPartToPyVista` converts the `ModelPart` in-memory — no disk I/O required.

In [ ]:
grid = pv_utils.ModelPartToPyVista(
    mp,
    useDeformedConfiguration=False,
    nodalVariables=[KM.PRESSURE, KM.DISPLACEMENT]
)

print("Converted PyVista Unstructured Grid:")
print(grid)

### Step 4: Visualizing the Mesh and Results

PyVista's built-in `.plot()` method on the converted grid provides quick interactive rendering.

In [ ]:
# Quick plot: PRESSURE scalar field on the undeformed mesh
grid.plot(
    scalars="PRESSURE",
    show_edges=True,
    cmap="viridis",
    cpos="xy",
    text="Kratos Undeformed Mesh - Nodal Pressure"
)

#### Using `PlotModelPart` for Rich Rendering

`PlotModelPart` is the high-level rendering function. It mirrors the design of FElupe's `Scene.plot()` and provides:

- **`name`** — variable to colour by (Kratos `Variable` or string).
- **`component`** — component index: `0/1/2` for X/Y/Z; `None` for L2 magnitude. Labels are auto-generated (`X`, `Y`, `Z`, `Magnitude`, `XX`/`YY`/`ZZ`/`XY`/`YZ`/`XZ` for Voigt-6, `(Max. Principal)` etc.).
- **`warpByVector`** — warp the mesh by a vector field (e.g. `KM.DISPLACEMENT`).
- **`factor`** — warp scaling factor.
- **`showUndeformed`** — ghost the original mesh at 20% opacity when warping.
- **`view`** — camera preset: `'xy'`, `'xz'`, `'yz'`, `'iso'`, or `'default'` (auto-selects `xy` + parallel projection for flat 2-D meshes).
- **`cmap`**, **`theme`**, **`showEdges`**, **`addAxes`** — standard rendering options.
- **`offScreen`** / **`notebook`** — headless rendering for scripts and Jupyter.
- **`plotter`** — pass an existing `pv.Plotter` to compose multiple objects in one scene.

The function returns a `pyvista.Plotter` (not yet shown). Call `.show()` to display interactively or `.screenshot()` to save an image.


In [ ]:
# Plot PRESSURE with auto-generated label and 2-D camera auto-detection
pv_utils.PlotModelPart(
    mp,
    name=KM.PRESSURE,
    cmap="viridis",
    showEdges=True,
    nodalVariables=[KM.PRESSURE]
).show()

In [ ]:
# Component selection: render individual components of the DISPLACEMENT vector.
# The scalar bar label is auto-generated: 'DISPLACEMENT X', 'DISPLACEMENT Y', etc.
for comp, comp_label in [(0, "X"), (1, "Y"), (None, "Magnitude")]:
    pv_utils.PlotModelPart(
        mp,
        name=KM.DISPLACEMENT,
        component=comp,
        cmap="plasma",
        showEdges=True,
    ).show()

### Step 5: Warp by Vector

`PlotModelPart` can warp the mesh in-place using any nodal vector variable.
When `warpByVector` is set:
- The undeformed mesh is shown as a 20%-opacity ghost (set `showUndeformed=False` to disable).
- The deformed mesh is rendered on top with the selected scalar.

The legacy `ComputeWarpedMesh` helper is still available for workflows that need the warped grid as a PyVista object.

In [ ]:
# PlotModelPart with integrated warp: no intermediate grid needed
pv_utils.PlotModelPart(
    mp,
    name=KM.PRESSURE,
    warpByVector=KM.DISPLACEMENT,
    factor=3.0,
    showUndeformed=True,
    cmap="plasma",
    showEdges=True,
).show()

In [ ]:
# Legacy approach: ComputeWarpedMesh returns the warped grid for further processing
warped_mesh = pv_utils.ComputeWarpedMesh(
    mp,
    KM.DISPLACEMENT,
    factor=1.5,
    nodalVariables=[KM.PRESSURE]
)

warped_mesh.plot(
    scalars="PRESSURE",
    show_edges=True,
    cmap="plasma",
    cpos="xy",
    text="Kratos Warped Mesh - Displaced Shape & Nodal Pressure"
)

#### Composing Multiple Objects in One Scene

Pass an existing `pv.Plotter` to `PlotModelPart` via the `plotter` argument to overlay multiple model parts or results in a single scene.
Each call mutates the plotter by calling `plotter.add_mesh()` and returns the same `plotter` for chaining.

In [ ]:
# Overlay: undeformed mesh (transparent) + warped coloured mesh
base_plotter = pv.Plotter()

# Add undeformed mesh as a light wireframe
pv_utils.PlotModelPart(mp, plotter=base_plotter, opacity=0.15, showEdges=True, addAxes=False)

# Overlay the warped mesh with PRESSURE colouring
pv_utils.PlotModelPart(
    mp,
    name=KM.PRESSURE,
    warpByVector=KM.DISPLACEMENT,
    factor=3.0,
    showUndeformed=False,
    cmap="viridis",
    showEdges=True,
    plotter=base_plotter,
).show()

### Step 6: Off-Screen Screenshot Export

`ScreenshotModelPart` renders the model off-screen and saves a PNG. All keyword arguments from `PlotModelPart` are forwarded, making it a drop-in for scripted pipelines.

Pass `filename=None` to get a raw NumPy image array instead of writing a file.

In [ ]:
import tempfile

# Save a screenshot of the pressure field
temp_dir = tempfile.gettempdir()
screenshot_path = os.path.join(temp_dir, "pressure_screenshot.png")

pv_utils.ScreenshotModelPart(
    mp,
    filename=screenshot_path,
    name=KM.PRESSURE,
    cmap="viridis",
    showEdges=True,
)

print(f"Screenshot saved to: {screenshot_path}")

# Display the saved image inline
from IPython.display import Image
Image(screenshot_path)

In [ ]:
# Alternatively, get the image array directly (no file written)
img_array = pv_utils.ScreenshotModelPart(
    mp,
    filename=None,
    name=KM.DISPLACEMENT,
    component=None,      # magnitude
    cmap="plasma",
    warpByVector=KM.DISPLACEMENT,
    factor=3.0,
)
print(f"Screenshot image array shape: {img_array.shape}  (height × width × RGBA)")

### Step 7: 3D Mesh — Setup

We construct a 3D block mesh of 8-node hexahedral elements (`Element3D8N`) with a quadratic temperature field and a steady velocity field.

In [ ]:
mp_3d = model.CreateModelPart("SimulationResults3D")
mp_3d.AddNodalSolutionStepVariable(KM.TEMPERATURE)
mp_3d.AddNodalSolutionStepVariable(KM.VELOCITY)

node_id = 1
for z in range(3):
    for y in range(3):
        for x in range(3):
            n = mp_3d.CreateNewNode(node_id, float(x), float(y), float(z))
            n.SetSolutionStepValue(KM.TEMPERATURE, 0, float(x**2 + y**2 + z**2))
            n.SetSolutionStepValue(KM.VELOCITY,    0, [2.0, 0.2 * y, -0.1 * x])
            node_id += 1

def get_node_id(gx, gy, gz):
    return 1 + gx + gy * 3 + gz * 9

properties_3d = mp_3d.GetProperties()[0]
elem_id = 1
for z in range(2):
    for y in range(2):
        for x in range(2):
            nodes = [
                get_node_id(x,     y,     z    ), get_node_id(x + 1, y,     z    ),
                get_node_id(x + 1, y + 1, z    ), get_node_id(x,     y + 1, z    ),
                get_node_id(x,     y,     z + 1), get_node_id(x + 1, y,     z + 1),
                get_node_id(x + 1, y + 1, z + 1), get_node_id(x,     y + 1, z + 1)
            ]
            mp_3d.CreateNewElement("Element3D8N", elem_id, nodes, properties_3d)
            elem_id += 1

print(f"3D Mesh created with {len(mp_3d.Nodes)} nodes and {len(mp_3d.Elements)} elements.")

3D Mesh created with 27 nodes and 8 elements.


In [ ]:
# Convert and keep the full 3-D grid for compositing later
grid_3d = pv_utils.ModelPartToPyVista(
    mp_3d,
    useDeformedConfiguration=False,
    nodalVariables=[KM.TEMPERATURE, KM.VELOCITY]
)

#### `PlotModelPart` on a 3-D Mesh

For 3-D meshes `PlotModelPart` auto-selects a perspective camera with a slight elevation/azimuth offset for a good default view.

In [ ]:
# Plot TEMPERATURE on the 3-D mesh using PlotModelPart
pv_utils.PlotModelPart(
    mp_3d,
    name=KM.TEMPERATURE,
    cmap="plasma",
    showEdges=True,
    nodalVariables=[KM.TEMPERATURE]
).show()

#### Surface Extraction

`CreateExtractedSurface` extracts the outer surface faces of a 3-D volumetric mesh as a `pyvista.PolyData` object. Nodal variables are transferred automatically.

In [ ]:
# Extract the outer surface of the 3-D block with TEMPERATURE transferred
surface = pv_utils.CreateExtractedSurface(
    mp_3d,
    nodalVariables=[KM.TEMPERATURE]
)

print(f"Surface: {surface.n_points} points, {surface.n_cells} cells")
print(f"Arrays:  {list(surface.point_data.keys())}")

surface.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="Outer Surface - Temperature"
)

#### Orthogonal Slices

In [ ]:
slices = pv_utils.CreateOrthogonalSlices(
    mp_3d, x=1.0, y=1.0, z=1.0,
    nodalVariables=[KM.TEMPERATURE]
)

slices.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="3D Orthogonal Slices - Temperature Nodal Field"
)

#### Isosurface (Contour) Extraction

In [ ]:
contours = pv_utils.CreateIsosurfaces(
    mp_3d, KM.TEMPERATURE, valuesOrNumber=4,
    nodalVariables=[KM.TEMPERATURE]
)

contours.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="3D Temperature Isosurfaces"
)

#### Fluid Flow Streamlines

In [ ]:
streamlines = pv_utils.CreateStreamlines(
    mp_3d,
    velocityVariable=KM.VELOCITY,
    sourceCenter=[1.0, 1.0, 1.0],
    sourceRadius=0.8,
    nPoints=30
)

plotter = pv.Plotter()
plotter.add_mesh(grid_3d.outline(), color="black")
plotter.add_mesh(streamlines, line_width=3.0, cmap="viridis", scalars="VELOCITY")
plotter.add_text("3D Velocity Flow Streamlines", font_size=12)
plotter.show()

#### Volumetric Scalar Thresholding

In [ ]:
threshed_grid = pv_utils.CreateThresholdedMesh(
    mp_3d, KM.TEMPERATURE,
    thresholdValue=4.0, thresholdType="above",
    nodalVariables=[KM.TEMPERATURE]
)

threshed_grid.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="3D Thresholded Domain (TEMPERATURE >= 4.0)"
)

#### Vector Field Glyphs (Arrow Plots)

In [ ]:
glyphs = pv_utils.CreateVectorGlyphs(
    mp_3d,
    vectorVariable=KM.VELOCITY,
    scaleFactor=0.4,
    glyphType="arrow",
    nodalVariables=[KM.VELOCITY]
)

plotter = pv.Plotter()
plotter.add_mesh(grid_3d.outline(), color="black")
plotter.add_mesh(glyphs, cmap="viridis", scalars="VELOCITY")
plotter.add_text("3D Nodal Velocity Vector Arrows", font_size=12)
plotter.show()

#### Mesh Clipping by a Plane

In [ ]:
clipped_grid = pv_utils.CreateClippedMesh(
    mp_3d,
    normal=[1.0, 0.0, 0.0],
    origin=[1.0, 1.0, 1.0],
    nodalVariables=[KM.TEMPERATURE]
)

clipped_grid.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="3D Clipped Mesh along X=1.0 Plane"
)

### Step 8: Exporting to File

`SaveModelPart` writes the mesh to a `.vtu` file compatible with ParaView.

In [ ]:
import tempfile
import os

temp_dir = tempfile.gettempdir()

# Save the warped mesh as a VTU file
output_vtu = os.path.join(temp_dir, "warped_simulation.vtu")
warped_mesh.save(output_vtu)
print(f"Mesh saved to: {output_vtu}")

# Save using the Kratos helper (preserves all exported variables)
output_mp = os.path.join(temp_dir, "model_part.vtu")
pv_utils.SaveModelPart(
    mp_3d,
    output_mp,
    nodalVariables=[KM.TEMPERATURE, KM.VELOCITY]
)
print(f"ModelPart saved to: {output_mp}")